# 제9장: 데이터 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch09.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### 데이터 거버넌스와 AI 거버넌스의 통합

## AI를 위한 데이터 요구사항

### 통합 거버넌스 아키텍처: 데이터 패브릭 관점

### 데이터 품질에서 AI 품질로의 전이

**데이터 품질-AI 품질 전이 분석기**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from enum import Enum
import numpy as np

class RiskLevel(Enum):
    CRITICAL = "Critical"
    HIGH = "High"
    MEDIUM = "Medium"
    LOW = "Low"

@dataclass
class DataQualityMetrics:
    """AI 학습 데이터 품질 지표"""
    completeness: float = 0.0      # 완전성 (0-1)
    label_accuracy: float = 0.0    # 레이블 정확도 (0-1)
    representation_score: float = 0.0  # 대표성 점수 (0-1)
    freshness_days: int = 0        # 데이터 신선도 (일)
    duplicate_ratio: float = 0.0   # 중복 비율 (0-1)
    pii_detected: bool = False     # PII 포함 여부

class QualityTransferAnalyzer:
    """데이터 품질 이슈의 AI 리스크 전이 분석기"""

    QUALITY_AI_RISK_MAP = {
        "completeness": {
            "ai_risk": "모델 성능 저하",
            "threshold": 0.95,
            "risk_level": RiskLevel.HIGH
        },
        "label_accuracy": {
            "ai_risk": "예측 신뢰도 하락",
            "threshold": 0.90,
            "risk_level": RiskLevel.HIGH
        },
        "representation_score": {
            "ai_risk": "공정성 위반 (편향)",
            "threshold": 0.80,
            "risk_level": RiskLevel.CRITICAL
        },
        "freshness_days": {
            "ai_risk": "컨셉 드리프트",
            "threshold": 90,
            "risk_level": RiskLevel.HIGH
        },
        "duplicate_ratio": {
            "ai_risk": "과적합, 검증 오염",
            "threshold": 0.05,
            "risk_level": RiskLevel.MEDIUM
        },
    }

    def analyze(self, metrics: DataQualityMetrics) -> Dict:
        """품질 지표를 분석하여 AI 리스크를 식별"""
        issues = []

        if metrics.completeness < self.QUALITY_AI_RISK_MAP["completeness"]["threshold"]:
            issues.append({
                "issue": "완전성 부족",
                "value": metrics.completeness,
                "ai_risk": "모델 성능 저하",
                "risk_level": RiskLevel.HIGH,
                "recommendation": "결측 패턴 분석 및 대체 전략 수립"
            })

        if metrics.representation_score < self.QUALITY_AI_RISK_MAP["representation_score"]["threshold"]:
            issues.append({
                "issue": "대표성 부족",
                "value": metrics.representation_score,
                "ai_risk": "공정성 위반",
                "risk_level": RiskLevel.CRITICAL,
                "recommendation": "민감 속성별 분포 검증 및 재샘플링"
            })

        if metrics.pii_detected:
            issues.append({
                "issue": "개인식별정보 포함",
                "value": True,
                "ai_risk": "프라이버시 위반, 법적 제재",
                "risk_level": RiskLevel.CRITICAL,
                "recommendation": "비식별화 처리 후 재검증"
            })

        overall_risk = RiskLevel.LOW
        if any(i["risk_level"] == RiskLevel.CRITICAL for i in issues):
            overall_risk = RiskLevel.CRITICAL
        elif any(i["risk_level"] == RiskLevel.HIGH for i in issues):
            overall_risk = RiskLevel.HIGH

        return {
            "total_issues": len(issues),
            "overall_risk": overall_risk,
            "issues": issues,
            "recommendation": "학습 진행 불가" if overall_risk == RiskLevel.CRITICAL
                             else "조건부 학습 허용"
        }

### 데이터시트(Datasheets)와 데이터 카드

### AI 학습 데이터 편향의 유형과 탐지

**학습 데이터 편향 탐지 프로토콜**

In [ ]:
from typing import Dict, List
import numpy as np
from scipy import stats

class DataBiasDetector:
    """
    학습 데이터 편향 탐지 프로토콜

    [실무 도구 9-2] 데이터 편향 탐지 프로토콜의 코드 구현
    민감 속성별 분포 검증, 레이블 편향 탐지, 대표성 평가를 수행
    """

    def __init__(self, sensitive_attributes: List[str]):
        self.sensitive_attributes = sensitive_attributes
        self.bias_report = {}

    def check_representation_bias(
        self,
        dataset_distribution: Dict[str, float],
        population_distribution: Dict[str, float],
        threshold: float = 0.8
    ) -> Dict:
        """
        표본 편향 탐지: 데이터셋 분포와 모집단 분포 비교

        각 하위 집단의 대표성 비율(Representation Ratio)이
        threshold 미만이면 편향으로 판정
        """
        results = {}
        for group, pop_ratio in population_distribution.items():
            data_ratio = dataset_distribution.get(group, 0.0)
            if pop_ratio > 0:
                representation_ratio = data_ratio / pop_ratio
            else:
                representation_ratio = float('inf')

            results[group] = {
                "dataset_ratio": data_ratio,
                "population_ratio": pop_ratio,
                "representation_ratio": representation_ratio,
                "bias_detected": representation_ratio < threshold
                                 or representation_ratio > (1 / threshold),
                "severity": "HIGH" if representation_ratio < 0.5 else
                           "MEDIUM" if representation_ratio < threshold else "LOW"
            }

        return results

    def check_label_bias(
        self,
        labels: np.ndarray,
        sensitive_attr: np.ndarray,
        positive_label: int = 1
    ) -> Dict:
        """
        레이블 편향 탐지: 민감 속성별 긍정 레이블 비율 비교

        통계적 유의성 검정(카이제곱 검정)을 통해 편향 판정
        """
        groups = np.unique(sensitive_attr)
        positive_rates = {}

        for group in groups:
            mask = sensitive_attr == group
            group_labels = labels[mask]
            positive_rate = np.mean(group_labels == positive_label)
            positive_rates[str(group)] = {
                "count": int(mask.sum()),
                "positive_rate": float(positive_rate)
            }

        # 카이제곱 검정으로 집단 간 차이 유의성 확인
        contingency = []
        for group in groups:
            mask = sensitive_attr == group
            pos = np.sum(labels[mask] == positive_label)
            neg = np.sum(labels[mask] != positive_label)
            contingency.append([pos, neg])

        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

        return {
            "positive_rates_by_group": positive_rates,
            "chi2_statistic": float(chi2),
            "p_value": float(p_value),
            "bias_detected": p_value < 0.05,
            "recommendation": "레이블링 프로세스 검토 필요"
                             if p_value < 0.05 else "유의미한 편향 미탐지"
        }

    def generate_bias_report(self, results: Dict) -> str:
        """편향 탐지 결과 보고서 생성"""
        report = "=" * 60 + "\n"
        report += "학습 데이터 편향 탐지 보고서\n"
        report += "=" * 60 + "\n\n"

        for check_name, result in results.items():
            report += f"[{check_name}]\n"
            bias_found = result.get("bias_detected", False)
            report += f"  편향 탐지: {'예' if bias_found else '아니오'}\n"
            if bias_found:
                report += f"  권고사항: {result.get('recommendation', 'N/A')}\n"
            report += "\n"

        return report

### 피처 스토어를 통한 데이터 거버넌스 강화

**피처 스토어 거버넌스 관리 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum

class FeatureStatus(Enum):
    DRAFT = "Draft"
    REVIEW = "Review"
    APPROVED = "Approved"
    DEPRECATED = "Deprecated"
    RETIRED = "Retired"

class SensitivityLevel(Enum):
    PUBLIC = "Public"
    INTERNAL = "Internal"
    CONFIDENTIAL = "Confidential"
    RESTRICTED = "Restricted"

@dataclass
class FeatureDefinition:
    """피처 정의 및 메타데이터"""
    feature_id: str
    name: str
    description: str
    data_type: str
    source_tables: List[str] = field(default_factory=list)
    transformation_logic: str = ""
    status: FeatureStatus = FeatureStatus.DRAFT
    sensitivity: SensitivityLevel = SensitivityLevel.INTERNAL
    owner: str = ""
    version: str = "1.0"
    created_at: datetime = field(default_factory=datetime.now)
    used_by_models: List[str] = field(default_factory=list)
    contains_pii: bool = False
    is_sensitive_attribute: bool = False

class FeatureStoreGovernance:
    """피처 스토어 거버넌스 관리 시스템"""

    def __init__(self):
        self.features: Dict[str, FeatureDefinition] = {}
        self.access_policies: Dict[str, List[str]] = {}
        self.audit_log: List[Dict] = []

    def register_feature(self, feature: FeatureDefinition) -> str:
        """피처 등록 및 초기 검증"""
        # PII 포함 피처는 자동으로 RESTRICTED로 분류
        if feature.contains_pii:
            feature.sensitivity = SensitivityLevel.RESTRICTED

        # 민감 속성 피처는 승인 워크플로 필수
        if feature.is_sensitive_attribute:
            feature.status = FeatureStatus.REVIEW

        self.features[feature.feature_id] = feature
        self._log("REGISTER", feature.feature_id, f"피처 등록: {feature.name}")
        return feature.feature_id

    def check_access(self, feature_id: str, model_id: str, user_role: str) -> bool:
        """피처 접근 권한 검증"""
        feature = self.features.get(feature_id)
        if not feature:
            return False

        # RESTRICTED 피처는 승인된 모델만 접근 가능
        if feature.sensitivity == SensitivityLevel.RESTRICTED:
            allowed = self.access_policies.get(feature_id, [])
            if model_id not in allowed:
                self._log("ACCESS_DENIED", feature_id,
                         f"모델 {model_id}의 RESTRICTED 피처 접근 거부")
                return False

        self._log("ACCESS_GRANTED", feature_id, f"모델 {model_id} 접근 허용")
        return True

    def get_feature_lineage(self, feature_id: str) -> Dict:
        """피처 계보 추적: 원천 데이터부터 사용 모델까지"""
        feature = self.features.get(feature_id)
        if not feature:
            return {}

        return {
            "feature_id": feature.feature_id,
            "source_tables": feature.source_tables,
            "transformation": feature.transformation_logic,
            "consuming_models": feature.used_by_models,
            "version": feature.version,
            "sensitivity": feature.sensitivity.value
        }

    def deprecate_feature(self, feature_id: str, reason: str) -> List[str]:
        """피처 폐기 시 영향받는 모델 식별"""
        feature = self.features.get(feature_id)
        if not feature:
            return []

        affected_models = feature.used_by_models.copy()
        feature.status = FeatureStatus.DEPRECATED
        self._log("DEPRECATE", feature_id,
                 f"피처 폐기: {reason}, 영향 모델: {affected_models}")
        return affected_models

    def _log(self, action: str, feature_id: str, detail: str):
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "action": action,
            "feature_id": feature_id,
            "detail": detail
        })

## 데이터 계보(Lineage)와 출처 관리

### 전통적 리니지에서 AI 통합 리니지로

### 리니지 그래프 모델링

**통합 리니지 관리 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set, Optional
from datetime import datetime
from enum import Enum

class AssetType(Enum):
    TABLE = "Table"
    FEATURE = "Feature"
    DATASET = "Dataset"
    MODEL = "Model"
    PREDICTION = "Prediction"
    PIPELINE = "Pipeline"

class RelationType(Enum):
    DERIVED_FROM = "derived_from"
    TRAINED_ON = "trained_on"
    PRODUCES = "produces"
    DEPENDS_ON = "depends_on"
    VERSION_OF = "version_of"

@dataclass
class LineageNode:
    """리니지 그래프의 노드 (데이터 자산)"""
    asset_id: str
    asset_type: AssetType
    name: str
    version: str = "1.0"
    metadata: Dict = field(default_factory=dict)

@dataclass
class LineageEdge:
    """리니지 그래프의 엣지 (의존 관계)"""
    source_id: str
    target_id: str
    relation: RelationType
    created_at: datetime = field(default_factory=datetime.now)

class IntegratedLineageManager:
    """데이터 패브릭 기반 통합 리니지 관리 시스템"""

    def __init__(self):
        self.nodes: Dict[str, LineageNode] = {}
        self.edges: List[LineageEdge] = []
        self.adjacency: Dict[str, List[str]] = {}  # 순방향
        self.reverse_adj: Dict[str, List[str]] = {}  # 역방향

    def register_asset(self, node: LineageNode):
        """데이터 자산 등록"""
        self.nodes[node.asset_id] = node
        if node.asset_id not in self.adjacency:
            self.adjacency[node.asset_id] = []
        if node.asset_id not in self.reverse_adj:
            self.reverse_adj[node.asset_id] = []

    def add_dependency(self, source_id: str, target_id: str,
                       relation: RelationType):
        """의존 관계 추가 (source -> target)"""
        edge = LineageEdge(source_id, target_id, relation)
        self.edges.append(edge)
        self.adjacency.setdefault(source_id, []).append(target_id)
        self.reverse_adj.setdefault(target_id, []).append(source_id)

    def impact_analysis(self, source_id: str) -> List[Dict]:
        """
        영향 분석: 특정 데이터 자산 변경 시 영향받는 모든 다운스트림 자산 식별
        BFS 알고리즘으로 그래프 탐색
        """
        visited = set()
        queue = [source_id]
        impacted = []

        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)

            for downstream in self.adjacency.get(current, []):
                if downstream not in visited:
                    queue.append(downstream)
                    node = self.nodes.get(downstream)
                    if node:
                        impacted.append({
                            "asset_id": node.asset_id,
                            "asset_type": node.asset_type.value,
                            "name": node.name,
                            "distance": len(visited)
                        })

        return impacted

    def root_cause_analysis(self, model_id: str) -> List[Dict]:
        """
        원인 분석: 모델의 모든 업스트림 데이터 자산을 역추적
        """
        visited = set()
        queue = [model_id]
        ancestors = []

        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)

            for upstream in self.reverse_adj.get(current, []):
                if upstream not in visited:
                    queue.append(upstream)
                    node = self.nodes.get(upstream)
                    if node:
                        ancestors.append({
                            "asset_id": node.asset_id,
                            "asset_type": node.asset_type.value,
                            "name": node.name
                        })

        return ancestors

    def find_affected_models(self, data_subject_id: str) -> List[str]:
        """
        GDPR 삭제 요청 대응: 특정 데이터 주체의 데이터가
        학습에 사용된 모델 식별
        """
        affected_models = []
        for node_id, node in self.nodes.items():
            if node.asset_type == AssetType.MODEL:
                ancestors = self.root_cause_analysis(node_id)
                for ancestor in ancestors:
                    if data_subject_id in ancestor.get("metadata", {}).get(
                        "data_subjects", []):
                        affected_models.append(node_id)
                        break
        return affected_models

### 메타데이터 기반 자동화

## 데이터 접근 제어와 프라이버시

### AI 파이프라인에서의 데이터 접근 제어

### 민감 데이터의 AI 학습 제한

**민감 데이터 AI 학습 적격성 검증기**

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class DataColumn:
    """데이터 컬럼 메타데이터"""
    name: str
    data_type: str
    is_pii: bool = False
    is_prohibited: bool = False
    is_proxy: bool = False
    proxy_for: str = ""
    consent_for_ai: bool = True
    pseudonymized: bool = False

class SensitiveDataValidator:
    """민감 데이터 AI 학습 적격성 검증기"""

    def validate_columns(self, columns: List[DataColumn]) -> Dict:
        """컬럼별 AI 학습 적격성 검증"""
        results = {
            "approved": [],
            "rejected": [],
            "conditional": [],
            "warnings": []
        }

        for col in columns:
            if col.is_prohibited:
                results["rejected"].append({
                    "column": col.name,
                    "reason": f"금지 속성 (AI 의사결정 직접 사용 불가)",
                    "action": "학습 데이터에서 제외"
                })
            elif col.is_pii and not col.pseudonymized:
                results["rejected"].append({
                    "column": col.name,
                    "reason": "비식별화 미처리 PII",
                    "action": "가명처리 후 재검증"
                })
            elif col.is_proxy:
                results["conditional"].append({
                    "column": col.name,
                    "reason": f"프록시 변수 ({col.proxy_for}의 대리)",
                    "action": "공정성 영향 평가 후 결정"
                })
            elif not col.consent_for_ai:
                results["rejected"].append({
                    "column": col.name,
                    "reason": "AI 학습 동의 미확보",
                    "action": "동의 확보 또는 학습 제외"
                })
            else:
                results["approved"].append({
                    "column": col.name,
                    "status": "학습 적격"
                })

        return results

### 머신 언러닝(Machine Unlearning)과 잊힐 권리

**머신 언러닝 거버넌스 관리 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum

class UnlearningMethod(Enum):
    EXACT_RETRAIN = "Exact Retraining"
    SISA = "SISA (Sharded Training)"
    GRADIENT_APPROX = "Gradient-based Approximation"
    CERTIFIED = "Certified Unlearning"

class UnlearningStatus(Enum):
    REQUESTED = "Requested"
    IN_PROGRESS = "In Progress"
    COMPLETED = "Completed"
    VERIFIED = "Verified"
    FAILED = "Failed"

@dataclass
class UnlearningRequest:
    """머신 언러닝 요청"""
    request_id: str
    data_subject_id: str
    legal_basis: str  # GDPR Art.17, 개인정보보호법 등
    affected_models: List[str] = field(default_factory=list)
    requested_at: datetime = field(default_factory=datetime.now)
    deadline: Optional[datetime] = None  # 법적 기한 (보통 30일)
    status: UnlearningStatus = UnlearningStatus.REQUESTED
    method: Optional[UnlearningMethod] = None
    verification_result: Optional[Dict] = None

class MachineUnlearningManager:
    """머신 언러닝 거버넌스 관리 시스템"""

    def __init__(self, lineage_manager):
        self.lineage_manager = lineage_manager
        self.requests: Dict[str, UnlearningRequest] = {}

    def submit_request(self, data_subject_id: str, legal_basis: str) -> str:
        """삭제 요청 접수 및 영향 분석"""
        # 리니지를 통해 영향받는 모델 식별
        affected_models = self.lineage_manager.find_affected_models(
            data_subject_id)

        request = UnlearningRequest(
            request_id=f"UNL-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            data_subject_id=data_subject_id,
            legal_basis=legal_basis,
            affected_models=affected_models,
        )

        self.requests[request.request_id] = request
        return request.request_id

    def select_method(self, request_id: str) -> UnlearningMethod:
        """최적의 언러닝 기법 선택"""
        request = self.requests[request_id]

        # 고위험 모델이 포함된 경우 인증된 언러닝 필수
        for model_id in request.affected_models:
            # 모델 리스크 등급 확인 로직
            pass

        # 영향 모델 수와 비용을 고려하여 기법 선택
        if len(request.affected_models) <= 3:
            return UnlearningMethod.EXACT_RETRAIN
        else:
            return UnlearningMethod.SISA

    def verify_unlearning(self, request_id: str) -> Dict:
        """언러닝 완료 검증"""
        request = self.requests[request_id]

        verification = {
            "request_id": request_id,
            "verified_at": datetime.now().isoformat(),
            "models_verified": [],
            "overall_result": "PASS"
        }

        for model_id in request.affected_models:
            # 멤버십 추론 공격(MIA)을 통한 검증
            # 삭제 대상 데이터가 모델에 잔존하는지 확인
            mia_result = self._membership_inference_test(
                model_id, request.data_subject_id)

            verification["models_verified"].append({
                "model_id": model_id,
                "mia_result": "NOT_MEMBER" if not mia_result else "MEMBER",
                "passed": not mia_result
            })

            if mia_result:
                verification["overall_result"] = "FAIL"

        request.verification_result = verification
        request.status = (UnlearningStatus.VERIFIED
                         if verification["overall_result"] == "PASS"
                         else UnlearningStatus.FAILED)

        return verification

    def _membership_inference_test(self, model_id: str,
                                    data_id: str) -> bool:
        """멤버십 추론 테스트 (MIA): 삭제 대상 데이터의 잔존 여부 확인"""
        # 실제 구현에서는 Shadow Model 기반 MIA를 수행
        return False  # False = 잔존하지 않음 (검증 통과)

## 외부 데이터 및 사전 학습 모델의 거버넌스

### 외부 데이터 소싱의 거버넌스

### 사전 학습 모델(Pre-trained Model)의 거버넌스

**사전 학습 모델 거버넌스 평가 시스템**

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
from enum import Enum

class ComplianceStatus(Enum):
    COMPLIANT = "Compliant"
    NON_COMPLIANT = "Non-Compliant"
    CONDITIONAL = "Conditional"
    UNDER_REVIEW = "Under Review"

@dataclass
class PretrainedModelProfile:
    """사전 학습 모델 프로파일"""
    model_name: str
    provider: str
    license_type: str
    allows_commercial: bool = False
    training_data_disclosed: bool = False
    input_data_retention: bool = False  # 입력 데이터 보존 여부
    input_used_for_training: bool = False  # 입력 데이터 재학습 활용
    supports_korean: bool = False
    has_safety_evaluation: bool = False
    deployment_options: List[str] = None  # ["api", "on-premise", "cloud"]

class PretrainedModelGovernance:
    """사전 학습 모델 거버넌스 평가 시스템"""

    def evaluate_compliance(
        self,
        profile: PretrainedModelProfile,
        intended_use: str,
        data_sensitivity: str
    ) -> Dict:
        """종합 거버넌스 평가"""
        checks = []

        # 1. 라이선스 적합성
        if not profile.allows_commercial and intended_use == "commercial":
            checks.append({
                "check": "라이선스",
                "status": ComplianceStatus.NON_COMPLIANT,
                "detail": "상업적 사용 라이선스 미허용"
            })
        else:
            checks.append({
                "check": "라이선스",
                "status": ComplianceStatus.COMPLIANT,
                "detail": "라이선스 조건 충족"
            })

        # 2. 데이터 주권
        if profile.input_used_for_training and data_sensitivity in ["confidential", "restricted"]:
            checks.append({
                "check": "데이터 주권",
                "status": ComplianceStatus.NON_COMPLIANT,
                "detail": "민감 데이터가 모델 재학습에 활용될 위험"
            })
        elif profile.input_data_retention:
            checks.append({
                "check": "데이터 주권",
                "status": ComplianceStatus.CONDITIONAL,
                "detail": "입력 데이터 보존 정책 검토 필요"
            })
        else:
            checks.append({
                "check": "데이터 주권",
                "status": ComplianceStatus.COMPLIANT,
                "detail": "데이터 주권 리스크 없음"
            })

        # 3. 투명성
        if not profile.training_data_disclosed:
            checks.append({
                "check": "투명성",
                "status": ComplianceStatus.CONDITIONAL,
                "detail": "학습 데이터 미공개 - 편향 평가 제한적"
            })

        # 종합 판정
        non_compliant = any(c["status"] == ComplianceStatus.NON_COMPLIANT
                           for c in checks)

        return {
            "model": profile.model_name,
            "overall": ComplianceStatus.NON_COMPLIANT if non_compliant
                      else ComplianceStatus.CONDITIONAL,
            "checks": checks,
            "recommendation": "도입 불가" if non_compliant
                             else "조건부 승인 - 추가 검토 필요"
        }

### 오픈소스 데이터셋과 모델의 라이선스 관리

## 통합 거버넌스 조직: CDO와 CAIO의 협업

### 역할 분담의 원칙

### 통합 거버넌스 협의체 운영

### 데이터 스튜어드의 AI 거버넌스 역할 확장

## 실무 도구

### 제9장 요약